# M3 · Retrieval avanzado y orquestación de la normalización terminológica

Este notebook ejecuta y evalúa el retrieval del sistema RAG sobre guías clínicas. La lógica vive en
módulos `.py`; aquí solo se llaman en orden:

1. Pruebas unitarias.
2. Configuración e índice Chroma, reconstruido y verificado contra los chunks JSON.
3. Validación y resumen del gold set.
4. Prueba de humo de las técnicas de búsqueda.
5. Experimento controlado: todos los sistemas sobre las mismas consultas.
6. Análisis por tipo de consulta y decisiones de la normalización.
7. Comparación con otro reranker (opcional).
8. Integración con la generación de respuestas.

La implementación parte del laboratorio S08 (BM25, fusión RRF con k = 60 y reranker cross-encoder
mMARCO) y lo adapta al corpus del proyecto. Las rutas, los modelos y los parámetros están en
`M3/retrieval/config_retrieval.yaml`; para usar el corpus definitivo basta con cambiar `corpus: mock`
por `corpus: real` en ese archivo.

El código se descarga del repositorio del proyecto al comienzo de cada sesión, así que siempre se
ejecuta la última versión publicada en la rama `RAMA`. Con `corpus: real`, los chunks y el manifiesto
se leen de Drive, donde los deja la ingesta, y el índice se reconstruye en el disco local a partir de
esos chunks. El gold set del corpus real está en el repositorio (`M3/retrieval/data/real/`).

## 0 · Código desde el repositorio

La primera vez se clona la rama; en ejecuciones siguientes se sincroniza con la versión publicada.
La sincronización descarta los cambios hechos a los archivos dentro de Colab: los cambios de código o
de configuración deben publicarse en el repositorio. Para probar otra configuración sin publicarla,
modifíquela en el diccionario `cfg` de la sección 2 (por ejemplo, `cfg['corpus'] = 'real'`).

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/luisNP21/Topicos-IA.git'
RAMA = 'dev'                        # rama con la versión a probar
RAIZ = '/content/Topicos-IA'

def git(*args, cwd=RAIZ):
    r = subprocess.run(['git', *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} falló:\n{r.stderr.strip()}")
    return r.stdout.strip()

if os.path.isdir(os.path.join(RAIZ, '.git')):
    git('fetch', 'origin', RAMA)
    git('checkout', '-B', RAMA, f'origin/{RAMA}')
    git('reset', '--hard', f'origin/{RAMA}')
else:
    git('clone', '--branch', RAMA, REPO_URL, RAIZ, cwd='/content')

%cd {RAIZ}
print(git('log', '-1', f'--format=Rama {RAMA} · commit %h · %ad · %s', '--date=short'))

In [ ]:
%pip install -q sentence-transformers chromadb rank_bm25 groq python-dotenv PyYAML

In [ ]:
import sys

for ruta in ['M3/retrieval', 'M3/generacion', 'M3/corpus']:
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

requeridos = ['M3/retrieval/retrieval.py', 'M3/retrieval/gold_set.py', 'M3/retrieval/config_retrieval.yaml', 'M3/retrieval/data/mock/corpus_mock.json',
              'M3/corpus/corpus_store.py', 'M3/corpus/embeddings.py', 'M3/corpus/config.yaml',
              'M3/generacion/contratos.py', 'M3/generacion/mocks.py', 'M3/generacion/generacion.py',
              'M3/generacion/orquestacion.py']
faltantes = [r for r in requeridos if not os.path.exists(r)]
if faltantes:
    raise FileNotFoundError(f'La rama {RAMA} no tiene: {faltantes}')
print('Archivos completos.')

## 1 · Pruebas unitarias

Verifican la lógica con modelos simulados, sin descargar nada.

In [ ]:
!cd M3/retrieval && python -m unittest test_retrieval
!cd M3/generacion && python -m unittest test_generacion

## 2 · Configuración e índice

La configuración se lee de `config_retrieval.yaml`. El índice Chroma se construye siempre en el disco
local de Colab a partir de los chunks JSON del perfil, con el mismo código de la ingesta
(`M3/corpus/embeddings.py` y `corpus_store.py`). Si ya existe y coincide con los chunks, se reutiliza;
si no, se reconstruye. Después se verifica que el número, los ids y los textos del índice coincidan
con los chunks JSON y, si difieren, la celda se detiene.

Con `corpus: mock` los chunks son un corpus de prueba de 13 guías con tres secciones cada una. Con
`corpus: real` se monta Drive, donde la ingesta deja los chunks y el manifiesto; del manifiesto solo
se usan las guías que están en el índice.

In [ ]:
from config_retrieval import (cargar_config, configurar_pipeline, indice_desde_config, normalizador_desde_config,
                              parametros_orquestacion, rutas)

cfg = cargar_config()
R = rutas(cfg)
print('Corpus:', cfg['corpus'], '| chunks:', R['corpus_json'], '| índice:', R['chroma_dir'], '| resultados:', R['salida'])

if cfg['corpus'] == 'real':
    from google.colab import drive
    drive.mount('/content/drive')          # los chunks y el manifiesto del corpus real están en Drive

indice = configurar_pipeline(cfg)          # reconstruye el índice si no coincide con los chunks y lo verifica

## 3 · Gold set

Las consultas y sus chunks relevantes se leen de las rutas del perfil activo en el YAML (`consultas`,
`anotaciones` y `gold_meta`). Antes de medir se comprueba que el gold set corresponda al índice
cargado: que existan todos los `chunk_id`, que cada chunk pertenezca a una guía relevante, que ninguna
consulta dentro del corpus quede sin chunks y que la huella del corpus coincida con la de
`gold_meta.json`. Si algo falla, la celda se detiene y dice qué comparación falló. El experimento
repite esta validación.

El gold set del corpus real lo anotó una IA simulando a un anotador humano, sin revisión humana.

In [ ]:
import pandas as pd
from gold_set import cargar_gold, resumen_gold, validar_gold

gold = cargar_gold(cfg)
display(pd.DataFrame(validar_gold(indice, gold)).set_index('Comprobación'))   # se detiene si no corresponde

resumen_del_gold = resumen_gold(gold)
print('Anotador:', resumen_del_gold['anotador'] or 'sin dato')
display(pd.DataFrame(resumen_del_gold['por_tipo']).set_index('Tipo de consulta'))
display(pd.DataFrame(resumen_del_gold['por_guia']).set_index('Guía'))
display(pd.DataFrame(resumen_del_gold['por_consulta']).set_index('Consulta'))

## 4 · Prueba de humo

Se busca una consulta del gold set por cada tipo de consulta.

In [ ]:
import collections
from retrieval import info_indice

info = info_indice(indice)
print(f"Índice: {info['n_chunks']} chunks, {info['n_documentos']} guías, huella {info['huella_sha256']}")
print('Secciones:', dict(collections.Counter(zip(indice.seccion_tipo, indice.seccion_origen))))

MODOS_HUMO = {'densa': 'Denso (ingenuo)', 'rerank': 'Híbrido + reranker',
              'intencion_sin_rerank': 'Tratamiento, sin reranker', 'intencion': 'Tratamiento + reranker',
              'intencion_seccion': 'Tratamiento + reranker + sección'}
humo = {}
for c in gold['consultas']:
    humo.setdefault(c['tipo'], c['consulta'])
for q in humo.values():
    print(f'\n{q}')
    for modo, nombre in MODOS_HUMO.items():
        top = indice.buscar(q, 3, modo)
        print(f'  {nombre:34}', [(f['chunk_id'].replace('mock_', ''), f['seccion_tipo'], round(f['score'], 3))
                                   for f in top])

Cada resultado muestra el chunk, el tipo de sección y el score. Conviene revisar tres cosas: que las
técnicas con consultas de tratamiento pongan primero un chunk de sección `tratamiento`, que las
consultas fuera del corpus reciban scores bajos del reranker y cuántos chunks se clasificaron
por título y cuántos por similitud. En el corpus real, los clasificados por similitud deben revisarse
a mano en una muestra.

## 5 · Experimento controlado

| Sistema | Qué agrega |
|---|---|
| Denso (RAG ingenuo) | Línea base de S07: similitud de embeddings |
| Híbrido BM25 + denso | Búsqueda léxica fusionada con RRF |
| Híbrido + reranker | Cross-encoder sobre los 30 mejores candidatos |
| Híbrido + reranker con pregunta de tratamiento | El reranker recibe «¿Cuál es el tratamiento de…?» |
| Consultas de tratamiento, sin reranker | Tres reformulaciones con intención fusionadas con RRF |
| Consultas de tratamiento + reranker | Las reformulaciones más el reranker con la pregunta |
| … + filtro de sección | El reranker solo ordena chunks de secciones de tratamiento |
| Sistema completo | Normalización terminológica y compuerta de evidencia sobre `orquestacion.sistema_base` del YAML |
| Sistema completo + regla de siglas | Normaliza siempre las siglas |

Las dos filas de la mitad (reranker con pregunta y consultas sin reranker) separan qué aporta cada
pieza de la query transformation. Las métricas «Tratamiento» exigen que llegue el chunk de
tratamiento, no solo la guía correcta.

Los umbrales del sistema completo se calibran con validación cruzada: cada consulta se evalúa con
umbrales calibrados en los otros pliegues. En el corpus de prueba, las etiquetas de sección provienen
de los mismos títulos que usa el filtro de sección, así que ese sistema tiene ventaja por construcción.

In [ ]:
import pandas as pd
from experimento_s08 import NOMBRE, TIPOS_CONSULTA, cargar_consultas, correr_desde_config, correr_experimento

detalle, resumen, calibracion = correr_desde_config(cfg, indice)

tabla = pd.DataFrame(resumen).drop(columns='Clave').set_index('Sistema')
display(tabla.T)

In [ ]:
cols = {'umbral': 'Umbral para normalizar', 'umbral_evidencia': 'Umbral de evidencia',
        'hay_zona_dudosa': 'Existe zona dudosa', 'evidencia_conservada': 'Evidencia correcta conservada'}
umbrales = pd.DataFrame([{'Calibración': f"Pliegue {p['pliegue']} (sin sus consultas)", **p} for p in calibracion['pliegues']]
                        + [{'Calibración': 'Final (todas las consultas)', **calibracion['final']}])
display(umbrales.set_index('Calibración')[list(cols)].rename(columns=cols))
print('Validación:', calibracion['validacion'])

Los umbrales finales son los que se configuran en el pipeline. Si varían mucho entre pliegues, la
calibración todavía depende de pocas consultas. «Existe zona dudosa» indica si el umbral de evidencia
quedó por debajo del de normalización; si no, el sistema solo distingue entre confiable y sin evidencia.

## 6 · Análisis por tipo de consulta

In [ ]:
df = pd.DataFrame(detalle)
orden = [NOMBRE[f['Clave']] for f in resumen]
dentro = df[~df.fuera_de_corpus]

def por_tipo(campo):
    t = dentro.pivot_table(index='tipo', columns='sistema', values=campo, aggfunc='mean')
    return t[[c for c in orden if c in t.columns]].rename(index=TIPOS_CONSULTA).rename_axis(index='Tipo de consulta',
                                                                                           columns=None).round(2).T

print('Tratamiento en 1.er lugar, por tipo de consulta')
display(por_tipo('hit1_sec'))
print('MRR de la guía, por tipo de consulta')
display(por_tipo('rr'))

In [ ]:
piezas = ['reranker', 'reranker_pregunta', 'intencion_sin_reranker', 'intencion']
aporte = tabla.loc[[NOMBRE[p] for p in piezas], ['Tratamiento en 1.er lugar', 'MRR del tratamiento',
                                                   'Latencia (ms por consulta)']]
display(aporte)

Esta tabla separa las dos piezas de la query transformation. Si «Consultas de tratamiento, sin
reranker» ya alcanza a «Consultas de tratamiento + reranker», la mejora viene de las reformulaciones y
el reranker no se gana su lugar; si queda por debajo, el reranker sí aporta. «Híbrido + reranker con
pregunta de tratamiento» mide el efecto de la pregunta sin las reformulaciones.

In [ ]:
decisiones = df[df.clave == 'completo_siglas'][['consulta', 'tipo', 'tool', 'query_final', 'hit1_sec', 'rechazo',
                                                'tool_reason']]
decisiones = decisiones.assign(tipo=decisiones.tipo.map(TIPOS_CONSULTA)).rename(columns={
    'consulta': 'Consulta', 'tipo': 'Tipo de consulta', 'tool': 'Usó la normalización', 'query_final': 'Consulta final',
    'hit1_sec': 'Tratamiento en 1.er lugar', 'rechazo': 'Sin fragmentos entregados', 'tool_reason': 'Motivo'})
display(decisiones.reset_index(drop=True))

## 7 · Otro reranker (opcional)

El reranker alternativo del YAML (`BAAI/bge-reranker-v2-m3`) es multilingüe y más grande. Conviene ejecutarlo con GPU
(Entorno de ejecución › Cambiar tipo de entorno de ejecución › T4).

In [ ]:
indice_bge = indice_desde_config(cfg, modelo_reranker=cfg['modelos']['reranker_alternativo'])
_, resumen_bge, _ = correr_experimento(indice_bge, cargar_consultas(R['consultas']), k=cfg['retrieval']['k'],
                                      claves=('ingenuo', 'reranker', 'intencion'))
comparacion = pd.concat([
    tabla.loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (mMARCO)'),
    pd.DataFrame(resumen_bge).drop(columns='Clave').set_index('Sistema')
      .loc[[NOMBRE['reranker'], NOMBRE['intencion']]].rename(index=lambda s: s + ' (bge-m3)'),
])
display(comparacion[['Guía correcta en 1.er lugar', 'Tratamiento en 1.er lugar', 'MRR del tratamiento',
                     'Latencia (ms por consulta)']])

## 8 · Integración con la generación

Se ejecuta la cadena completa para tres entidades: orquestación, retrieval y generación de la
respuesta. La generación recibe como parámetro la función que llama al LLM; aquí se usa una función
simulada que devuelve un texto fijo, para verificar el recorrido de los datos sin consumir la cuota de
la API. Para obtener respuestas reales se reemplaza por `GroqGenerator()` del módulo `generacion`. La
normalización es la simulada del perfil activo (`mapa_normalizacion` en el YAML); con
`mapa_normalizacion: null` se usa la herramienta real, que necesita `BIOPORTAL_API_KEY`.

In [ ]:
from retrieval import pregunta_intencion, retrieve_advanced
from orquestacion import resolver_query
from generacion import generar_respuesta

parametros = parametros_orquestacion(cfg)       # umbrales calibrados (o provisionales si aún no hay)
normalizar = normalizador_desde_config(cfg)     # mapa simulado del YAML, o la herramienta real si es null

def generador_simulado(system, user):
    return 'Respuesta simulada con la fuente citada.'

entidades = (['neumonía adquirida en la comunidad', 'HTA', 'amiloidosis'] if cfg['corpus'] == 'mock'
             else ['diabetes mellitus tipo 2', 'DM2', 'amiloidosis'])
for entidad in entidades:
    q = resolver_query(entidad, retrieve_fn=retrieve_advanced, normalizar_fn=normalizar, **parametros)
    r = generar_respuesta(q['query_final'], q['fragments'], generar_fn=generador_simulado,
                          pregunta=pregunta_intencion(q['query_final']))
    print(f"\n{entidad}")
    print('  normalización:', 'sí' if q['tool_invoked'] else 'no', '|', q['tool_reason'] or 'score suficiente')
    print('  fragmentos:', [(f['chunk_id'], f['seccion_tipo'], f['score']) for f in q['fragments']])
    print('  respuesta:', r['answer'], '| sin evidencia' if r['fallback_used'] else '')

## 9 · Descargar los resultados

Los archivos generados en Colab se pierden al cerrar la sesión. Esta celda descarga la carpeta de
resultados comprimida, para revisarla o publicarla en el repositorio.

In [ ]:
import shutil
from google.colab import files

archivo = shutil.make_archive('/content/resultados_retrieval', 'zip', R['salida'])
files.download(archivo)